# Medical Emergency Assistant - Gemma 4 E4B Fine-Tuning

Fine-tune Google's Gemma 4 E4B model with QLoRA for concise, effective medical emergency responses.
Optimized for RTX 5070 Ti (16GB VRAM) and edge deployment.

## Setup

In [ ]:
# Skip PyTorch install on Colab (already has CUDA-enabled PyTorch)
# If running locally, uncomment:
# %pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

In [ ]:
%pip install git+https://github.com/huggingface/transformers.git
%pip install git+https://github.com/huggingface/peft.git
%pip install git+https://github.com/huggingface/trl.git
%pip install accelerate>=1.5.0 datasets scipy matplotlib huggingface_hub bitsandbytes>=0.45.0

In [ ]:
import torch
import time
import json
import numpy as np
import os

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from datasets import load_dataset, Dataset
from datetime import datetime
from pathlib import Path

import bitsandbytes as bnb
print(f"bitsandbytes version: {bnb.__version__}")

ts = datetime.now().strftime("%Y%m%d_%H%M%S")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"CUDA version: {torch.version.cuda}")
    print(f"bf16 supported: {torch.cuda.is_bf16_supported()}")
else:
    raise RuntimeError("CUDA not available — 4-bit quantization requires a GPU.")

In [ ]:
# Path setup
root = Path.cwd()
output_dir = root / "outputs"
output_dir.mkdir(exist_ok=True)

# HuggingFace login (needed for Gemma gated model)
# Use token= to avoid interactive prompt freezing on Windows
from huggingface_hub import login
login(token=os.environ.get("HF_TOKEN"))  # Set HF_TOKEN env var, or replace with your token string

## Dataset Preparation

Load training and validation data from JSONL files. Each line should be a JSON object with a `messages` array:
```jsonl
{"messages": [{"role": "system", "content": "..."}, {"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]}
```

Place your files at:
- `data/medical_train.jsonl` — training examples (aim for 200+)
- `data/medical_val.jsonl` — validation examples (10-20% of training size)

In [ ]:
SYSTEM_PROMPT = "You are an expert emergency medical dispatcher. Provide a concise, direct diagnosis and immediate action steps with no filler."

# Load from JSONL files
data_dir = root / "data"
train_path = data_dir / "medical_train.jsonl"
val_path = data_dir / "medical_val.jsonl"

assert train_path.is_file(), f"Training data not found at {train_path}"
assert val_path.is_file(), f"Validation data not found at {val_path}"

train_dataset = load_dataset("json", data_files=str(train_path), split="train")
val_dataset = load_dataset("json", data_files=str(val_path), split="train")

print(f"Train: {len(train_dataset)} examples")
print(f"Val:   {len(val_dataset)} examples")
print(f"\n--- First example preview ---")
print(f"User: {train_dataset[0]['messages'][1]['content'][:120]}...")
print(f"Assistant: {train_dataset[0]['messages'][2]['content'][:120]}...")

## Load Gemma 4 E4B with 4-bit Quantization

Gemma 4 E4B is a ~4B parameter model. With QLoRA (4-bit quantization), it fits
comfortably in ~10GB VRAM, leaving headroom on the RTX 5070 Ti (16GB).

In [ ]:
MODEL_ID = "google/gemma-3n-E4B-it"

# 4-bit quantization config for QLoRA
# Use float16 compute dtype (T4 doesn't fully support bf16)
use_bf16 = torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
print("Tokenizer loaded. Now loading model (this downloads several GB on first run)...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=compute_dtype,
)

print(f"Model loaded: {MODEL_ID}")
print(f"Using dtype: {compute_dtype}")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## Test Base Model (Before Fine-Tuning)

In [ ]:
def generate_response(model, tokenizer, prompt, system_prompt=None, max_new_tokens=512):
    """Generate a response using chat template."""
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})

    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    start = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
        )
    elapsed = time.time() - start

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)

    tokens_generated = len(new_tokens)
    print(f"[{tokens_generated} tokens in {elapsed:.1f}s = {tokens_generated/elapsed:.1f} tok/s]")
    return response

In [ ]:
test_prompt = "Adult male, 55, clutching chest, sweating profusely, pain radiating to left arm. Started 10 minutes ago."

print("=== BASE MODEL RESPONSE (before fine-tuning) ===")
base_response = generate_response(model, tokenizer, test_prompt, system_prompt=SYSTEM_PROMPT)
print(base_response)

## Format Dataset for Training

In [ ]:
def format_example(example):
    """Format messages into a single text field using the chat template."""
    text = tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)
    return {"text": text}

train_dataset = train_dataset.map(format_example)
val_dataset = val_dataset.map(format_example)

print("--- Formatted example ---")
print(train_dataset["text"][0][:500])

## QLoRA Setup

In [ ]:
# Prepare model for QLoRA training
# use_gradient_checkpointing saves VRAM during backward pass
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## Training

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=str(output_dir / "gemma4-medical-qlora"),
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=1,
    eval_strategy="epoch",
    save_strategy="epoch",
    report_to="none",
    remove_unused_columns=False,
    dataset_text_field="text",
    max_seq_length=1024,
    bf16=use_bf16,
    fp16=not use_bf16,
    optim="paged_adamw_8bit",
    gradient_checkpointing=False,  # already enabled manually in QLoRA setup
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

trainer.train()

In [ ]:
# Save adapter v1
final_path = output_dir / "gemma4-medical-qlora"
trainer.model.save_pretrained(f"{final_path}/{ts}_final_adapter_v1")
tokenizer.save_pretrained(f"{final_path}/{ts}_final_adapter_v1")
print(f"Adapter saved to: {final_path}/{ts}_final_adapter_v1")

## Evaluation v1

In [ ]:
# Plot training loss
import matplotlib.pyplot as plt

log_history = trainer.state.log_history

train_steps = [x["step"] for x in log_history if "loss" in x]
train_loss = [x["loss"] for x in log_history if "loss" in x]
eval_epochs = [x["epoch"] for x in log_history if "eval_loss" in x]
eval_loss = [x["eval_loss"] for x in log_history if "eval_loss" in x]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_steps, train_loss, marker="o", markersize=3)
ax1.set_xlabel("Step")
ax1.set_ylabel("Loss")
ax1.set_title("Training Loss")
ax1.grid(True)

ax2.plot(eval_epochs, eval_loss, marker="x", color="orange")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Eval Loss")
ax2.set_title("Evaluation Loss")
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
print("=== FINE-TUNED MODEL RESPONSE (v1 - 3 epochs) ===")
response_v1 = generate_response(model, tokenizer, test_prompt, system_prompt=SYSTEM_PROMPT)
print(response_v1)

In [ ]:
# Evaluate on test scenarios
test_scenarios = [
    "Person having a seizure, shaking on the ground for 2 minutes.",
    "Deep cut on the leg, heavy bleeding, remote location.",
    "Child choking on food, turning blue, cannot cough.",
    "Person collapsed after marathon, confused, hot skin, not sweating.",
    "Elderly person fell, hip pain, cannot stand or move leg.",
]

print("=== Evaluation on test scenarios ===")
for i, scenario in enumerate(test_scenarios):
    print(f"\n{'='*60}")
    print(f"Scenario {i+1}: {scenario}")
    print(f"{'='*60}")
    resp = generate_response(model, tokenizer, scenario, system_prompt=SYSTEM_PROMPT)
    print(resp)

## Extended Training (v2 - 6 epochs)

In [ ]:
trainer.args.num_train_epochs = 6
trainer.train()

In [ ]:
# Save adapter v2
trainer.model.save_pretrained(f"{final_path}/{ts}_final_adapter_v2")
tokenizer.save_pretrained(f"{final_path}/{ts}_final_adapter_v2")
print(f"Adapter v2 saved to: {final_path}/{ts}_final_adapter_v2")

## Evaluation v2

In [ ]:
# Plot updated losses
log_history = trainer.state.log_history

train_steps = [x["step"] for x in log_history if "loss" in x]
train_loss = [x["loss"] for x in log_history if "loss" in x]
eval_epochs = [x["epoch"] for x in log_history if "eval_loss" in x]
eval_loss = [x["eval_loss"] for x in log_history if "eval_loss" in x]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_steps, train_loss, marker="o", markersize=3)
ax1.set_xlabel("Step")
ax1.set_ylabel("Loss")
ax1.set_title("Training Loss (6 epochs)")
ax1.grid(True)

ax2.plot(eval_epochs, eval_loss, marker="x", color="orange")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Eval Loss")
ax2.set_title("Evaluation Loss (6 epochs)")
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
print("=== FINE-TUNED MODEL RESPONSE (v2 - 6 epochs) ===")
response_v2 = generate_response(model, tokenizer, test_prompt, system_prompt=SYSTEM_PROMPT)
print(response_v2)

In [ ]:
# Re-evaluate on test scenarios
print("=== Evaluation v2 on test scenarios ===")
for i, scenario in enumerate(test_scenarios):
    print(f"\n{'='*60}")
    print(f"Scenario {i+1}: {scenario}")
    print(f"{'='*60}")
    resp = generate_response(model, tokenizer, scenario, system_prompt=SYSTEM_PROMPT)
    print(resp)

## Export for Snapdragon X Elite Deployment

Merge the LoRA adapter back into the base model, then convert for on-device inference
on a Snapdragon X Elite laptop (Hexagon NPU + Adreno GPU + Oryon CPU).

Recommended inference runtimes for Snapdragon X Elite:
- **llama.cpp** (GGUF format) — best general-purpose option, ARM-optimized
- **Qualcomm AI Engine Direct / QNN SDK** — leverages the Hexagon NPU for best perf
- **ONNX Runtime** with QNN execution provider — good middle ground

In [ ]:
from peft import AutoPeftModelForCausalLM

# Reload and merge (need full precision for merging)
merged_model = AutoPeftModelForCausalLM.from_pretrained(
    f"{final_path}/{ts}_final_adapter_v2",
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
merged_model = merged_model.merge_and_unload()

# Save merged HF model
merged_path = output_dir / "gemma4-medical-merged"
merged_model.save_pretrained(str(merged_path))
tokenizer.save_pretrained(str(merged_path))
print(f"Merged model saved to: {merged_path}")

# --- Conversion options for Snapdragon X Elite ---

print("\n=== Option 1: GGUF via llama.cpp (recommended starting point) ===")
print("On the Snapdragon laptop, llama.cpp runs well on the Oryon CPU (ARM NEON).")
print("Steps:")
print(f"  1. git clone https://github.com/ggml-org/llama.cpp && cd llama.cpp")
print(f"  2. pip install -r requirements.txt")
print(f"  3. python convert_hf_to_gguf.py {merged_path} --outtype q4_K_M --outfile gemma4-medical-q4km.gguf")
print(f"  4. Copy .gguf to Snapdragon laptop")
print(f"  5. ./llama-server -m gemma4-medical-q4km.gguf -c 1024 --port 8080")
print()
print("Q4_K_M gives the best quality/size tradeoff (~2.5GB file, fits easily in X Elite's RAM).")
print()
print("=== Option 2: ONNX Runtime + QNN (Hexagon NPU acceleration) ===")
print("For maximum performance using the dedicated NPU:")
print(f"  1. pip install optimum[exporters]")
print(f"  2. optimum-cli export onnx --model {merged_path} gemma4-medical-onnx/")
print(f"  3. On Snapdragon laptop: pip install onnxruntime-qnn")
print(f"  4. Use QNN execution provider for Hexagon NPU inference")
print()
print("=== Option 3: Qualcomm AI Hub ===")
print("Qualcomm AI Hub can compile and optimize models directly for Snapdragon X Elite.")
print("  1. Upload merged model to aihub.qualcomm.com")
print("  2. Select Snapdragon X Elite as target device")
print("  3. Download optimized model binary")

## Next Steps

### 1. Scale Up the Dataset
The seed examples above are minimal. For production quality:
- Expand to 200-500+ examples covering diverse emergencies
- Include trauma, cardiac, respiratory, neurological, environmental, toxicological scenarios
- Have medical professionals review and validate responses
- Add edge cases: pediatric, geriatric, pregnancy-related emergencies

### 2. Increase LoRA Rank
If the model struggles to learn the structured output format:
- Try `r=32` or `r=64` (monitor VRAM usage)
- Increase `lora_alpha` proportionally

### 3. Target More Modules
For deeper adaptation, add MLP layers:
```python
target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
```

### 4. Snapdragon X Elite Deployment Notes
- **RAM**: X Elite has 16-64GB unified RAM — the Q4_K_M quantized model (~2.5GB) leaves plenty of room
- **llama.cpp on ARM**: Build with `cmake -B build -DLLAMA_NATIVE=ON` to enable Oryon-specific optimizations
- **NPU path**: For best latency, explore QNN SDK to offload to the Hexagon NPU — requires ONNX or QNN model format
- **Benchmark target**: Aim for <1s first-token latency and >20 tok/s generation for usable emergency response times
- **Offline-first**: Package the model + runtime as a self-contained app — no internet dependency